In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
import warnings
warnings.filterwarnings('ignore')


###Reading Data

In [8]:
df=pd.read_csv(r'/content/heart_failure_clinical_records_dataset.csv',encoding = "latin-1")

In [9]:
df.head()

,age,anaemia,creatinine_phosphokinase,diabetes,ejection_fraction,high_blood_pressure,platelets,serum_creatinine,serum_sodium,sex,smoking,time,DEATH_EVENT
0,75.0,0,582,0,20,1,265000.00,1.9,130,1,0,4,1
1,55.0,0,7861,0,38,0,263358.03,1.1,136,1,0,6,1
2,65.0,0,146,0,20,0,162000.00,1.3,129,1,1,7,1
3,50.0,1,111,0,20,0,210000.00,1.9,137,1,0,7,1
4,65.0,1,160,1,20,0,327000.00,2.7,116,0,0,8,1


In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 299 entries, 0 to 298
Data columns (total 13 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   age                       299 non-null    float64
 1   anaemia                   299 non-null    int64  
 2   creatinine_phosphokinase  299 non-null    int64  
 3   diabetes                  299 non-null    int64  
 4   ejection_fraction         299 non-null    int64  
 5   high_blood_pressure       299 non-null    int64  
 6   platelets                 299 non-null    float64
 7   serum_creatinine          299 non-null    float64
 8   serum_sodium              299 non-null    int64  
 9   sex                       299 non-null    int64  
 10  smoking                   299 non-null    int64  
 11  time                      299 non-null    int64  
 12  DEATH_EVENT               299 non-null    int64  
dtypes: float64(3), int64(10)
memory usage: 30.5 KB


###Split input &output

In [11]:
input=df.drop('DEATH_EVENT',axis=1)
output=df['DEATH_EVENT']

###Train and Test For Data

In [15]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(input,output,test_size=0.2,random_state=7)

In [58]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler

n_scaler = MinMaxScaler()

numerical_feature = Pipeline(steps=[("scaler", MinMaxScaler())])

x_train_nscaled = numerical_feature.fit_transform(x_train)
x_test_nscaled = numerical_feature.transform(x_test)

In [59]:
numerical_feature

Pipeline(steps=[('scaler', MinMaxScaler())])

In [19]:
x_train_nscaled = pd.DataFrame(x_train_nscaled, columns=n_scaler.get_feature_names_out())


In [21]:
print(x_train_nscaled.head())

        age  anaemia  creatinine_phosphokinase  diabetes  ejection_fraction  \
0  0.375764      1.0                  0.016331       1.0           0.393939   
1  0.327273      0.0                  0.015438       1.0           0.363636   
2  0.109091      0.0                  0.018500       1.0           0.045455   
3  0.000000      0.0                  0.008548       0.0           0.318182   
4  0.636364      1.0                  0.022965       1.0           0.363636   

   high_blood_pressure  platelets  serum_creatinine  serum_sodium  sex  \
0                  1.0   0.191781          0.056180      0.696970  0.0   
1                  1.0   0.348692          0.022472      0.878788  0.0   
2                  1.0   0.278954          0.179775      0.333333  0.0   
3                  0.0   0.259029          0.067416      0.696970  1.0   
4                  1.0   0.293898          0.011236      0.545455  1.0   

   smoking      time  
0      0.0  0.597865  
1      0.0  0.281139  
2      0.0 

###ML

In [22]:
from sklearn.linear_model import LogisticRegression
lr=LogisticRegression()
lr.fit(x_train_nscaled,y_train)

LogisticRegression()

In [40]:
y_pred_lr=lr.predict(x_test_nscaled)

In [41]:
from sklearn.metrics import accuracy_score
accuracy_score(y_test,y_pred_lr)

0.9

In [42]:
from sklearn .metrics import classification_report
print(classification_report(y_test,y_pred_lr))

              precision    recall  f1-score   support

           0       0.90      0.98      0.94        46
           1       0.90      0.64      0.75        14

    accuracy                           0.90        60
   macro avg       0.90      0.81      0.84        60
weighted avg       0.90      0.90      0.89        60



In [34]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV
knn=KNeighborsClassifier()
param_grid={'n_neighbors':np.arange(1,10)}
knn_cv=GridSearchCV(knn,param_grid,cv=5)
knn_cv.fit(x_train_nscaled,y_train)
y_pred=knn_cv.predict(x_test_nscaled)


In [27]:
knn_cv.best_params_

{'n_neighbors': np.int64(2)}

In [35]:
accuracy_score(y_test,y_pred)

0.8

In [43]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV
dt=DecisionTreeClassifier(criterion="gini")
param_grid={'max_depth':np.arange(3,10),"min_samples_split":range(2,15)}
dt_cv=GridSearchCV(dt,param_grid)
dt_cv.fit(x_train_nscaled,y_train)
y_pred_dt=dt_cv.predict(x_test_nscaled)


In [30]:
print(dt_cv.best_score_,dt_cv.best_params_)

0.7906914893617021 {'max_depth': np.int64(8), 'min_samples_split': 6}


In [44]:
accuracy_score(y_test,y_pred_dt)

0.9

In [45]:
from sklearn .metrics import classification_report
print(classification_report(y_test,y_pred_dt))

              precision    recall  f1-score   support

           0       0.92      0.96      0.94        46
           1       0.83      0.71      0.77        14

    accuracy                           0.90        60
   macro avg       0.88      0.84      0.85        60
weighted avg       0.90      0.90      0.90        60



###Pipeline

In [63]:
final_pipeline = Pipeline(steps=[("numerical_feature",MinMaxScaler()) ,('Modeling', LogisticRegression())])
final_pipeline

Pipeline(steps=[('numerical_feature', MinMaxScaler()),
                ('Modeling', LogisticRegression())])

In [64]:
final_pipeline.fit(x_train,y_train)

Pipeline(steps=[('numerical_feature', MinMaxScaler()),
                ('Modeling', LogisticRegression())])

In [65]:
y_pred = final_pipeline.predict(x_test)
y_pred

array([0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1,
       0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0,
       0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0])

In [66]:
print('evaluation on training \n', classification_report(y_train, final_pipeline.predict(x_train)))
print('evaluation on testing \n', classification_report(y_test, y_pred))

evaluation on training 
               precision    recall  f1-score   support

           0       0.84      0.92      0.88       157
           1       0.82      0.67      0.74        82

    accuracy                           0.84       239
   macro avg       0.83      0.80      0.81       239
weighted avg       0.84      0.84      0.83       239

evaluation on testing 
               precision    recall  f1-score   support

           0       0.90      0.98      0.94        46
           1       0.90      0.64      0.75        14

    accuracy                           0.90        60
   macro avg       0.90      0.81      0.84        60
weighted avg       0.90      0.90      0.89        60



In [67]:
from sklearn.metrics import accuracy_score

acc = accuracy_score(y_test, y_pred)
print("Accuracy:", acc)


Accuracy: 0.9


In [68]:
import joblib
joblib.dump(final_pipeline,'final_model.pkl')

['final_model.pkl']

In [69]:
model = joblib.load("final_model.pkl")

In [70]:
model.predict(x_test)

array([0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1,
       0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0,
       0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0])